In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Simple Use

In [5]:
ctc_loss = nn.CTCLoss(
    blank=0,
    reduction="mean",
    zero_infinity=True
)

def call_ctc_loss(logits, targets):
    log_probs = F.log_softmax(logits, dim=-1)
    
    input_lengths = torch.tensor([ logits.shape[0] ], dtype=torch.long)
    target_lengths = torch.tensor([ targets.numel() ], dtype=torch.long)  
    
    loss = ctc_loss(
        log_probs,
        targets,
        input_lengths,
        target_lengths
    ) 
    return loss.item()

### Manual

In [3]:
def manual_ctc_loss(logits, targets):
    log_probs = F.log_softmax(logits, dim=-1)
    log_probs = log_probs[:, 0, :]

    T, V = log_probs.shape
    S = targets.numel()
    blank = 0

    extended = torch.full(
        (2 * S + 1,),
        blank,
        dtype=targets.dtype
    )
    extended[1::2] = targets

    S_ext = extended.numel()

    alpha = torch.full(
        (T, S_ext),
        float("-inf")
    )

    alpha[0, 0] = log_probs[0, extended[0]]

    if S_ext > 1:
        alpha[0, 1] = log_probs[0, extended[1]]

    for t in range(1, T):
        for s in range(S_ext):
            candidates = [alpha[t - 1, s]]

            if s - 1 >= 0:
                candidates.append(alpha[t - 1, s - 1])

            if s - 2 >= 0:
                current = extended[s]
                previous_previous = extended[s - 2]

                if current != blank and current != previous_previous:
                    candidates.append(alpha[t - 1, s - 2])

            alpha[t, s] = (
                torch.logsumexp(
                    torch.stack(candidates),
                    dim=0
                )
                + log_probs[t, extended[s]]
            )

    log_likelihood = torch.logsumexp(
        torch.stack([
            alpha[T - 1, S_ext - 1],
            alpha[T - 1, S_ext - 2]
        ]),
        dim=0
    )

    return (-log_likelihood).item()/targets.numel()

In [6]:
# vocab:
# 0 = blank
# 1 = A
# 2 = B
# 3 = C
# 4 = D

logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=0 -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # t=1 -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=2 -> Blank
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # t=3 -> B
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=4 -> Blank
])
targets = torch.tensor([1, 2], dtype=torch.long)

manual_ctc_loss(logits, targets), call_ctc_loss(logits, targets)

(0.00036331178853288293, 0.00036331178853288293)

In [7]:
logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=0 -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # t=1 -> A
    [[0.0, 0.0, 0.0, 10.0, 0.0]],  # t=2 -> C
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # t=3 -> B
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=4 -> Blank
])

targets = torch.tensor([1, 2], dtype=torch.long)

manual_ctc_loss(logits, targets), call_ctc_loss(logits, targets)

(4.451086521148682, 4.451086521148682)

In [8]:
logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=0 -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # t=1 -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=2 -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # t=3 -> A
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=4 -> Blank
])

targets = torch.tensor([1, 1], dtype=torch.long)

manual_ctc_loss(logits, targets), call_ctc_loss(logits, targets)

(0.0004087284323759377, 0.0004087284323759377)

In [9]:
logits = torch.tensor([
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=0 -> Blank
    [[0.0, 10.0, 0.0, 0.0, 0.0]],  # t=1 -> A
    [[0.0, 0.0, 10.0, 0.0, 0.0]],  # t=2 -> B
    [[0.0, 0.0, 0.0, 10.0, 0.0]],  # t=3 -> C
    [[10.0, 0.0, 0.0, 0.0, 0.0]],  # t=4 -> Blank
])

targets = torch.tensor([1, 2, 3], dtype=torch.long)

manual_ctc_loss(logits, targets), call_ctc_loss(logits, targets)

(0.00027248562158395845, 0.0002724856312852353)

In [10]:
# vocab:
# 0 = blank
# 1 = A
# 2 = B
# 3 = C
# 4 = D

logits = torch.tensor([
    [[8.2, 1.1, 0.4, 0.7, 0.2]],   # t=0  -> Blank
    [[0.8, 7.5, 1.2, 0.3, 0.9]],   # t=1  -> A
    [[6.9, 1.4, 0.6, 1.0, 0.2]],   # t=2  -> Blank
    [[0.5, 1.1, 8.0, 0.7, 0.3]],   # t=3  -> B
    [[7.7, 0.4, 1.0, 0.8, 0.2]],   # t=4  -> Blank
    [[0.9, 0.6, 7.3, 8.4, 0.5]],   # t=5  -> C
    [[8.1, 0.7, 0.3, 1.1, 0.6]],   # t=6  -> Blank
    [[0.4, 0.8, 1.0, 0.5, 7.8]],   # t=7  -> D
    [[7.4, 0.9, 0.5, 0.8, 1.2]],   # t=8  -> Blank
    [[0.6, 8.1, 0.7, 1.0, 0.3]],   # t=9  -> A
    [[7.9, 0.5, 0.8, 0.4, 1.1]],   # t=10 -> Blank
    [[0.7, 1.0, 8.3, 0.6, 0.4]],   # t=11 -> B
    [[7.2, 0.8, 0.4, 1.3, 0.6]],   # t=12 -> Blank
    [[0.5, 0.9, 0.7, 7.6, 1.0]],   # t=13 -> C
    [[8.4, 0.6, 0.9, 0.5, 0.3]],   # t=14 -> Blank
    [[7.5, 1.2, 0.8, 0.4, 0.9]],   # t=15 -> Blank
    [[0.7, 8.0, 0.5, 1.1, 0.4]],   # t=16 -> A
    [[8.2, 0.6, 0.9, 0.3, 0.7]],   # t=17 -> Blank
    [[0.4, 0.8, 7.7, 1.0, 0.5]],   # t=18 -> B
    [[7.8, 0.5, 0.7, 1.2, 0.4]],   # t=19 -> Blank
])

targets = torch.tensor(
    [1, 2, 3, 4, 1, 2, 3],
    dtype=torch.long
)

manual_ctc_loss(logits, targets), call_ctc_loss(logits, targets)

(2.1340721675327847, 2.1340720653533936)